# Offline GEO-filtered RAG over PDFs — V2 Notebook

This notebook adds the planned V2 improvements to the working GEO-filtered RAG pipeline:

- OCR fallback helpers for scanned/image PDFs
- incremental ingestion registry
- incremental ingestion that updates only affected GEO indexes
- threshold-based retrieval
- hybrid retrieval (vector + BM25)
- reranker integration
- deduplication
- comparison prompt template
- structured source output
- logging utilities

## Data layout

```text
data/
├── APAC/
│   ├── file1.pdf
│   └── file2.pdf
├── EMEA/
│   └── file3.pdf
└── AMER/
    └── file4.pdf
```

## Storage layout

```text
storage/
├── faiss_apac/
├── faiss_emea/
├── faiss_amer/
├── chunks_apac.jsonl
├── chunks_emea.jsonl
├── chunks_amer.jsonl
├── ingestion_registry.json
└── rag_pipeline.log
```

## What this notebook does

- Reads PDFs with `PyPDFLoader`
- Falls back to OCR when text extraction is too weak
- Attaches GEO metadata to every page and chunk
- Deduplicates chunks
- Stores a registry of processed PDFs
- Detects **new**, **updated**, and **deleted** PDFs
- Rebuilds only the GEO indexes that changed
- Uses hybrid retrieval:
  - FAISS vector retrieval
  - BM25 keyword retrieval
  - reciprocal-rank fusion
  - optional reranker
- Uses a comparison prompt when the query asks about multiple GEOs

In [ ]:
# Install Python packages if needed.
# Restart the kernel after installation if this is your first run.

# %pip install -qU langchain langchain-community langchain-text-splitters langchain-huggingface #     faiss-cpu pypdf sentence-transformers transformers accelerate torch tqdm rank-bm25 pdf2image pytesseract

## OCR runtime note

For OCR fallback, you also need local OCR/PDF tools installed on your machine:

- **Tesseract OCR**
- **Poppler** utilities

`pdf2image` converts PDF pages into images, and `pytesseract` reads text from those images.

In [1]:
from __future__ import annotations

import hashlib
import json
import logging
import math
import re
import shutil
from collections import defaultdict
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

from tqdm.auto import tqdm

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

from transformers import pipeline

In [2]:
@dataclass
class Settings:
    data_dir: Path = Path("data")
    storage_dir: Path = Path("storage")
    geos: Tuple[str, ...] = ("APAC", "EMEA", "AMER")

    # Chunking
    chunk_size: int = 800
    chunk_overlap: int = 100

    # OCR fallback
    min_page_chars: int = 40
    min_doc_chars_for_text_loader: int = 250
    min_usable_page_ratio: float = 0.30
    ocr_dpi: int = 200
    ocr_lang: str = "eng"

    # Incremental ingestion
    registry_filename: str = "ingestion_registry.json"

    # Embeddings / models
    embedding_model: str = "sentence-transformers/all-mpnet-base-v2"
    llm_model: str = "Qwen/Qwen2.5-3B-Instruct"
    reranker_model: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"

    # Generation
    max_new_tokens: int = 180
    do_sample: bool = False
    temperature: float = 0.0

    # Retrieval
    retrieval_fetch_k: int = 10
    top_k: int = 4
    score_threshold: float = 1.15         # lower is stricter for normalized L2 vector scores
    bm25_score_threshold: float = 0.05
    use_hybrid_retrieval: bool = True
    use_reranker: bool = True
    max_chars_per_chunk_in_prompt: int = 1200
    reranker_max_chars: int = 1200

    # RRF
    rrf_k: int = 60

settings = Settings()
settings.storage_dir.mkdir(parents=True, exist_ok=True)

settings

Settings(data_dir=WindowsPath('data'), storage_dir=WindowsPath('storage'), geos=('APAC', 'EMEA', 'AMER'), chunk_size=800, chunk_overlap=100, min_page_chars=40, min_doc_chars_for_text_loader=250, min_usable_page_ratio=0.3, ocr_dpi=200, ocr_lang='eng', registry_filename='ingestion_registry.json', embedding_model='sentence-transformers/all-mpnet-base-v2', llm_model='Qwen/Qwen2.5-3B-Instruct', reranker_model='cross-encoder/ms-marco-MiniLM-L-6-v2', max_new_tokens=180, do_sample=False, temperature=0.0, retrieval_fetch_k=10, top_k=4, score_threshold=1.15, bm25_score_threshold=0.05, use_hybrid_retrieval=True, use_reranker=True, max_chars_per_chunk_in_prompt=1200, reranker_max_chars=1200, rrf_k=60)

## Logging utilities

In [3]:
def setup_logger(log_file: Path) -> logging.Logger:
    logger = logging.getLogger("geo_rag_v2")
    logger.setLevel(logging.INFO)

    if logger.handlers:
        return logger

    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    file_handler = logging.FileHandler(log_file, encoding="utf-8")
    file_handler.setFormatter(formatter)

    stream_handler = logging.StreamHandler()
    stream_handler.setFormatter(formatter)

    logger.addHandler(file_handler)
    logger.addHandler(stream_handler)
    logger.propagate = False
    return logger


logger = setup_logger(settings.storage_dir / "rag_pipeline.log")
logger.info("Logger initialized")

2026-03-31 00:07:32 | INFO | Logger initialized


## Core helper functions

In [4]:
def normalize_geo(value: str) -> str:
    value = value.strip().upper()
    if value not in settings.geos:
        raise ValueError(f"Unsupported GEO: {value}")
    return value


def detect_geos_from_query(query: str) -> List[str]:
    text = query.upper()
    found = []

    for geo in settings.geos:
        if re.search(rf"\b{geo}\b", text):
            found.append(geo)

    return found


def is_comparison_query(query: str, geos: Optional[List[str]] = None) -> bool:
    text = query.lower()
    if geos and len(geos) > 1:
        return True

    comparison_terms = [
        "compare",
        "comparison",
        "difference",
        "different",
        "vs",
        "versus",
        "similarities",
        "differences",
    ]
    return any(term in text for term in comparison_terms)


def make_doc_id(path: Path) -> str:
    return hashlib.md5(str(path.resolve()).encode("utf-8")).hexdigest()


def clean_text(text: str) -> str:
    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def normalize_text_for_hash(text: str) -> str:
    text = clean_text(text).lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def page_has_usable_text(text: str, min_chars: int = settings.min_page_chars) -> bool:
    return len(clean_text(text)) >= min_chars


def build_chunk_id(source: str, page: Optional[int], idx: int) -> str:
    raw = f"{source}|{page}|{idx}"
    return hashlib.md5(raw.encode("utf-8")).hexdigest()


def chunk_text_hash(text: str) -> str:
    return hashlib.md5(normalize_text_for_hash(text).encode("utf-8")).hexdigest()


def tokenize_for_bm25(text: str) -> List[str]:
    return re.findall(r"\b\w+\b", text.lower())

## OCR fallback helpers

In [5]:
def ocr_pdf_to_documents(pdf_path: Path, geo: str) -> List[Document]:
    try:
        from pdf2image import convert_from_path
        import pytesseract
    except ImportError as e:
        raise ImportError(
            "OCR fallback requires pdf2image and pytesseract. "
            "Install them, and ensure poppler + tesseract are installed on the system."
        ) from e

    logger.info(f"OCR fallback started for {pdf_path.name}")
    images = convert_from_path(str(pdf_path), dpi=settings.ocr_dpi)
    doc_id = make_doc_id(pdf_path)

    ocr_docs: List[Document] = []
    for page_idx, image in enumerate(images):
        text = pytesseract.image_to_string(image, lang=settings.ocr_lang)
        cleaned = clean_text(text)
        if not page_has_usable_text(cleaned):
            continue

        metadata = {
            "geo": geo,
            "source": str(pdf_path),
            "doc_id": doc_id,
            "file_name": pdf_path.name,
            "page": page_idx,
            "loader": "ocr",
        }
        ocr_docs.append(Document(page_content=cleaned, metadata=metadata))

    logger.info(f"OCR fallback completed for {pdf_path.name}: {len(ocr_docs)} usable pages")
    return ocr_docs


def load_pdf_with_fallback(pdf_path: Path, geo: str) -> List[Document]:
    loader = PyPDFLoader(str(pdf_path))
    pages = loader.load()
    doc_id = make_doc_id(pdf_path)

    page_docs: List[Document] = []
    total_chars = 0
    usable_pages = 0

    for page_doc in pages:
        raw_text = page_doc.page_content or ""
        cleaned = clean_text(raw_text)
        total_chars += len(cleaned)

        if page_has_usable_text(cleaned):
            usable_pages += 1

        metadata = dict(page_doc.metadata)
        metadata.update(
            {
                "geo": geo,
                "source": str(pdf_path),
                "doc_id": doc_id,
                "file_name": pdf_path.name,
                "page": metadata.get("page"),
                "loader": "pypdf",
            }
        )
        page_docs.append(Document(page_content=cleaned, metadata=metadata))

    usable_ratio = usable_pages / max(1, len(page_docs))

    if total_chars < settings.min_doc_chars_for_text_loader or usable_ratio < settings.min_usable_page_ratio:
        logger.info(
            f"Falling back to OCR for {pdf_path.name} "
            f"(total_chars={total_chars}, usable_ratio={usable_ratio:.2f})"
        )
        return ocr_pdf_to_documents(pdf_path, geo)

    # keep only usable pages
    usable_docs = [doc for doc in page_docs if page_has_usable_text(doc.page_content)]
    logger.info(f"Loaded {pdf_path.name} with PyPDFLoader: {len(usable_docs)} usable pages")
    return usable_docs

## Chunking, deduplication, and chunk store helpers

In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=settings.chunk_size,
    chunk_overlap=settings.chunk_overlap,
)


def chunk_documents(docs: List[Document]) -> List[Document]:
    chunks = text_splitter.split_documents(docs)

    for idx, chunk in enumerate(chunks):
        source = chunk.metadata.get("source", "")
        page = chunk.metadata.get("page")
        chunk.metadata["chunk_id"] = build_chunk_id(source, page, idx)
        chunk.metadata["text_hash"] = chunk_text_hash(chunk.page_content)

    return chunks


def deduplicate_chunks(docs: List[Document]) -> List[Document]:
    unique_docs: List[Document] = []
    seen_hashes = set()

    for doc in docs:
        text_hash = doc.metadata.get("text_hash") or chunk_text_hash(doc.page_content)
        if text_hash in seen_hashes:
            continue
        seen_hashes.add(text_hash)
        doc.metadata["text_hash"] = text_hash
        unique_docs.append(doc)

    return unique_docs


def chunk_store_path(geo: str) -> Path:
    return settings.storage_dir / f"chunks_{geo.lower()}.jsonl"


def serialize_document(doc: Document) -> Dict[str, Any]:
    return {
        "page_content": doc.page_content,
        "metadata": doc.metadata,
    }


def deserialize_document(payload: Dict[str, Any]) -> Document:
    return Document(
        page_content=payload["page_content"],
        metadata=payload["metadata"],
    )


def save_chunk_store(geo: str, docs: List[Document]) -> None:
    path = chunk_store_path(geo)
    with path.open("w", encoding="utf-8") as f:
        for doc in docs:
            f.write(json.dumps(serialize_document(doc), ensure_ascii=False) + "\n")
    logger.info(f"Saved chunk store for {geo}: {len(docs)} chunks")


def load_chunk_store(geo: str) -> List[Document]:
    path = chunk_store_path(geo)
    if not path.exists():
        return []

    docs: List[Document] = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                docs.append(deserialize_document(json.loads(line)))
    return docs

## Incremental ingestion registry

In [7]:
def registry_path() -> Path:
    return settings.storage_dir / settings.registry_filename


def load_registry() -> Dict[str, Dict[str, Any]]:
    path = registry_path()
    if not path.exists():
        return {}

    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def save_registry(registry: Dict[str, Dict[str, Any]]) -> None:
    path = registry_path()
    with path.open("w", encoding="utf-8") as f:
        json.dump(registry, f, indent=2, ensure_ascii=False)
    logger.info(f"Saved registry with {len(registry)} entries")


def compute_file_hash(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def scan_pdf_inventory(data_dir: Path) -> Dict[str, Dict[str, Any]]:
    inventory: Dict[str, Dict[str, Any]] = {}

    for geo in settings.geos:
        geo_dir = data_dir / geo
        if not geo_dir.exists():
            continue

        for pdf_path in sorted(geo_dir.rglob("*.pdf")):
            stat = pdf_path.stat()
            inventory[str(pdf_path.resolve())] = {
                "path": str(pdf_path.resolve()),
                "geo": geo,
                "mtime": stat.st_mtime,
                "size": stat.st_size,
            }

    return inventory


def plan_ingestion_changes(
    inventory: Dict[str, Dict[str, Any]],
    registry: Dict[str, Dict[str, Any]],
) -> Dict[str, Any]:
    inventory_paths = set(inventory.keys())
    registry_paths = set(registry.keys())

    new_paths = sorted(inventory_paths - registry_paths)
    deleted_paths = sorted(registry_paths - inventory_paths)

    maybe_updated_paths = []
    unchanged_paths = []

    for path, current in inventory.items():
        if path not in registry:
            continue

        prior = registry[path]
        if current["mtime"] != prior.get("mtime") or current["size"] != prior.get("size"):
            maybe_updated_paths.append(path)
        else:
            unchanged_paths.append(path)

    updated_paths = []
    false_alarm_paths = []

    for path in maybe_updated_paths:
        current_hash = compute_file_hash(Path(path))
        if current_hash != registry[path].get("sha256"):
            updated_paths.append(path)
        else:
            false_alarm_paths.append(path)

    unchanged_paths.extend(false_alarm_paths)

    by_geo = {geo: {"new": [], "updated": [], "deleted": [], "unchanged": []} for geo in settings.geos}

    for path in new_paths:
        by_geo[inventory[path]["geo"]]["new"].append(path)

    for path in updated_paths:
        by_geo[inventory[path]["geo"]]["updated"].append(path)

    for path in deleted_paths:
        deleted_geo = registry[path]["geo"]
        by_geo[deleted_geo]["deleted"].append(path)

    for path in unchanged_paths:
        geo = inventory[path]["geo"]
        by_geo[geo]["unchanged"].append(path)

    touched_geos = [
        geo for geo, bucket in by_geo.items()
        if bucket["new"] or bucket["updated"] or bucket["deleted"]
    ]

    summary = {
        "touched_geos": touched_geos,
        "total_new": len(new_paths),
        "total_updated": len(updated_paths),
        "total_deleted": len(deleted_paths),
        "total_unchanged": len(unchanged_paths),
        "by_geo": by_geo,
    }
    return summary

In [8]:
def log_change_plan(change_plan: Dict[str, Any]) -> None:
    logger.info(
        "Change plan | new=%s updated=%s deleted=%s unchanged=%s touched_geos=%s",
        change_plan["total_new"],
        change_plan["total_updated"],
        change_plan["total_deleted"],
        change_plan["total_unchanged"],
        change_plan["touched_geos"],
    )
    for geo, bucket in change_plan["by_geo"].items():
        logger.info(
            "GEO=%s | new=%s updated=%s deleted=%s unchanged=%s",
            geo,
            len(bucket["new"]),
            len(bucket["updated"]),
            len(bucket["deleted"]),
            len(bucket["unchanged"]),
        )

## Build embeddings

In [9]:
embeddings = HuggingFaceEmbeddings(
    model_name=settings.embedding_model,
    encode_kwargs={"normalize_embeddings": True},
)
embeddings

HuggingFaceEmbeddings(model_name='sentence-transformers/all-mpnet-base-v2', cache_folder=None, model_kwargs={}, encode_kwargs={'normalize_embeddings': True}, query_encode_kwargs={}, multi_process=False, show_progress=False)

## Incremental ingestion + per-GEO index rebuild

In [10]:
def geo_index_path(geo: str) -> Path:
    return settings.storage_dir / f"faiss_{geo.lower()}"


def remove_geo_index_if_exists(geo: str) -> None:
    index_dir = geo_index_path(geo)
    if index_dir.exists():
        shutil.rmtree(index_dir)
        logger.info(f"Removed FAISS index for {geo}")


def save_geo_index(geo: str, docs: List[Document], embeddings_model: HuggingFaceEmbeddings) -> None:
    if not docs:
        remove_geo_index_if_exists(geo)
        return

    index_dir = geo_index_path(geo)
    index_dir.mkdir(parents=True, exist_ok=True)
    vectorstore = FAISS.from_documents(docs, embeddings_model)
    vectorstore.save_local(str(index_dir))
    logger.info(f"Saved FAISS index for {geo}: {len(docs)} chunks")


def load_geo_vectorstores(embeddings_model: HuggingFaceEmbeddings) -> Dict[str, FAISS]:
    stores: Dict[str, FAISS] = {}
    for geo in settings.geos:
        index_dir = geo_index_path(geo)
        if index_dir.exists():
            stores[geo] = FAISS.load_local(
                str(index_dir),
                embeddings_model,
                allow_dangerous_deserialization=True,
            )
    return stores


def upsert_registry_entry(registry: Dict[str, Dict[str, Any]], pdf_path: Path, geo: str) -> None:
    stat = pdf_path.stat()
    registry[str(pdf_path.resolve())] = {
        "path": str(pdf_path.resolve()),
        "geo": geo,
        "mtime": stat.st_mtime,
        "size": stat.st_size,
        "sha256": compute_file_hash(pdf_path),
        "doc_id": make_doc_id(pdf_path),
        "last_ingested_at": datetime.utcnow().isoformat() + "Z",
    }


def remove_registry_entry(registry: Dict[str, Dict[str, Any]], path: str) -> None:
    if path in registry:
        registry.pop(path, None)


def rebuild_touched_geos_only(
    change_plan: Dict[str, Any],
    registry: Dict[str, Dict[str, Any]],
    embeddings_model: HuggingFaceEmbeddings,
) -> Dict[str, FAISS]:
    touched_geos = change_plan["touched_geos"]

    for geo in touched_geos:
        bucket = change_plan["by_geo"][geo]
        changed_paths = bucket["new"] + bucket["updated"]
        deleted_paths = bucket["deleted"]
        touched_paths = set(changed_paths + deleted_paths)

        existing_chunks = load_chunk_store(geo)
        remaining_chunks = [
            doc for doc in existing_chunks
            if doc.metadata.get("source") not in touched_paths
        ]

        logger.info(
            f"{geo}: existing_chunks={len(existing_chunks)} remaining_after_remove={len(remaining_chunks)} "
            f"changed_files={len(changed_paths)} deleted_files={len(deleted_paths)}"
        )

        new_page_docs: List[Document] = []
        for path_str in tqdm(changed_paths, desc=f"Ingesting {geo} changes"):
            pdf_path = Path(path_str)
            page_docs = load_pdf_with_fallback(pdf_path, geo)
            new_page_docs.extend(page_docs)
            upsert_registry_entry(registry, pdf_path, geo)

        for path_str in deleted_paths:
            remove_registry_entry(registry, path_str)

        new_chunk_docs = deduplicate_chunks(chunk_documents(new_page_docs))
        combined_chunks = deduplicate_chunks(remaining_chunks + new_chunk_docs)

        save_chunk_store(geo, combined_chunks)
        save_geo_index(geo, combined_chunks, embeddings_model)

        logger.info(
            f"{geo}: rebuilt with {len(combined_chunks)} chunks "
            f"(new_chunks={len(new_chunk_docs)}, deleted_files={len(deleted_paths)})"
        )

    save_registry(registry)
    return load_geo_vectorstores(embeddings_model)

In [11]:
# Run incremental ingestion

registry = load_registry()
inventory = scan_pdf_inventory(settings.data_dir)
change_plan = plan_ingestion_changes(inventory, registry)
log_change_plan(change_plan)

geo_vectorstores = rebuild_touched_geos_only(change_plan, registry, embeddings)

# If nothing changed, just load the existing indexes
if not change_plan["touched_geos"]:
    geo_vectorstores = load_geo_vectorstores(embeddings)

print("Loaded vectorstores for GEOs:", list(geo_vectorstores.keys()))

2026-03-31 00:08:05 | INFO | Change plan | new=5 updated=0 deleted=0 unchanged=0 touched_geos=['APAC', 'EMEA', 'AMER']
2026-03-31 00:08:05 | INFO | GEO=APAC | new=2 updated=0 deleted=0 unchanged=0
2026-03-31 00:08:05 | INFO | GEO=EMEA | new=2 updated=0 deleted=0 unchanged=0
2026-03-31 00:08:05 | INFO | GEO=AMER | new=1 updated=0 deleted=0 unchanged=0
2026-03-31 00:08:05 | INFO | APAC: existing_chunks=0 remaining_after_remove=0 changed_files=2 deleted_files=0


Ingesting APAC changes:   0%|          | 0/2 [00:00<?, ?it/s]

2026-03-31 00:08:06 | INFO | Loaded 2.pdf with PyPDFLoader: 105 usable pages
C:\Users\rahul\AppData\Local\Temp\ipykernel_30060\2934998790.py:46: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_ingested_at": datetime.utcnow().isoformat() + "Z",
2026-03-31 00:08:06 | INFO | Loaded 5deb6b48-5d7f-4a37-a361-d01317ac4bb7.pdf with PyPDFLoader: 11 usable pages
2026-03-31 00:08:06 | INFO | Saved chunk store for APAC: 157 chunks
2026-03-31 00:08:09 | INFO | Saved FAISS index for APAC: 157 chunks
2026-03-31 00:08:09 | INFO | APAC: rebuilt with 157 chunks (new_chunks=157, deleted_files=0)
2026-03-31 00:08:09 | INFO | EMEA: existing_chunks=0 remaining_after_remove=0 changed_files=2 deleted_files=0


Ingesting EMEA changes:   0%|          | 0/2 [00:00<?, ?it/s]

2026-03-31 00:08:11 | INFO | Loaded 092ea5dd-f471-4578-b6cb-9fcb17a9b84a.pdf with PyPDFLoader: 32 usable pages
2026-03-31 00:08:11 | INFO | Loaded 48e0858a-4dec-4761-a782-7e9570da73a5.pdf with PyPDFLoader: 47 usable pages
2026-03-31 00:08:11 | INFO | Saved chunk store for EMEA: 161 chunks
2026-03-31 00:08:15 | INFO | Saved FAISS index for EMEA: 161 chunks
2026-03-31 00:08:15 | INFO | EMEA: rebuilt with 161 chunks (new_chunks=161, deleted_files=0)
2026-03-31 00:08:15 | INFO | AMER: existing_chunks=0 remaining_after_remove=0 changed_files=1 deleted_files=0


Ingesting AMER changes:   0%|          | 0/1 [00:00<?, ?it/s]

2026-03-31 00:08:16 | INFO | Loaded 11.pdf with PyPDFLoader: 17 usable pages
2026-03-31 00:08:16 | INFO | Saved chunk store for AMER: 43 chunks
2026-03-31 00:08:16 | INFO | Saved FAISS index for AMER: 43 chunks
2026-03-31 00:08:16 | INFO | AMER: rebuilt with 43 chunks (new_chunks=43, deleted_files=0)
2026-03-31 00:08:16 | INFO | Saved registry with 5 entries


Loaded vectorstores for GEOs: ['APAC', 'EMEA', 'AMER']


## Build BM25 indexes from the chunk stores

In [14]:
def build_bm25_indexes() -> Dict[str, Dict[str, Any]]:
    try:
        from rank_bm25 import BM25Okapi
    except ImportError as e:
        raise ImportError("Hybrid retrieval requires rank-bm25. Install it first.") from e

    indexes: Dict[str, Dict[str, Any]] = {}

    for geo in settings.geos:
        docs = load_chunk_store(geo)
        if not docs:
            continue

        tokenized_corpus = [tokenize_for_bm25(doc.page_content) for doc in docs]
        bm25 = BM25Okapi(tokenized_corpus)
        indexes[geo] = {
            "bm25": bm25,
            "docs": docs,
        }
        logger.info(f"Built BM25 index for {geo}: {len(docs)} chunks")

    return indexes


bm25_indexes = build_bm25_indexes()
print("BM25 indexes:", list(bm25_indexes.keys()))

2026-03-31 00:09:29 | INFO | Built BM25 index for APAC: 157 chunks
2026-03-31 00:09:29 | INFO | Built BM25 index for EMEA: 161 chunks
2026-03-31 00:09:29 | INFO | Built BM25 index for AMER: 43 chunks


BM25 indexes: ['APAC', 'EMEA', 'AMER']


## Load the local instruction model and optional reranker

In [15]:
generator = pipeline(
    "text-generation",
    model=settings.llm_model,
    torch_dtype="auto",
    device_map="auto",
)

generator

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cpu


In [16]:
def load_reranker():
    try:
        from sentence_transformers import CrossEncoder
    except ImportError as e:
        raise ImportError(
            "Reranker requires sentence-transformers. Install it first."
        ) from e

    reranker = CrossEncoder(settings.reranker_model)
    logger.info(f"Loaded reranker: {settings.reranker_model}")
    return reranker


reranker = load_reranker() if settings.use_reranker else None
reranker

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

C:\Users\rahul\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rahul\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

2026-03-31 00:09:54 | INFO | Loaded reranker: cross-encoder/ms-marco-MiniLM-L-6-v2


CrossEncoder(
  (model): BertForSequenceClassification(
    (bert): BertModel(
      (embeddings): BertEmbeddings(
        (word_embeddings): Embedding(30522, 384, padding_idx=0)
        (position_embeddings): Embedding(512, 384)
        (token_type_embeddings): Embedding(2, 384)
        (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): BertEncoder(
        (layer): ModuleList(
          (0-5): 6 x BertLayer(
            (attention): BertAttention(
              (self): BertSdpaSelfAttention(
                (query): Linear(in_features=384, out_features=384, bias=True)
                (key): Linear(in_features=384, out_features=384, bias=True)
                (value): Linear(in_features=384, out_features=384, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (output): BertSelfOutput(
                (dense): Linear(in_features=384, out_features=384,

## Threshold-based + hybrid retrieval + reranking

In [17]:
def inspect_retrieval_scores(question: str, stores: Dict[str, FAISS], k: int = 5) -> None:
    geos = detect_geos_from_query(question) or list(stores.keys())
    print(f"Query GEOs: {geos}")

    for geo in geos:
        if geo not in stores:
            continue

        print(f"\n--- {geo} ---")
        docs_and_scores = stores[geo].similarity_search_with_score(question, k=k)
        for i, (doc, score) in enumerate(docs_and_scores, start=1):
            print(
                f"{i}. score={score:.4f} | file={doc.metadata.get('file_name')} | "
                f"page={doc.metadata.get('page')} | chunk_id={doc.metadata.get('chunk_id')}"
            )

In [18]:
def bm25_search(question: str, geo: str, bm25_indexes: Dict[str, Dict[str, Any]], k: int) -> List[Tuple[Document, float]]:
    if geo not in bm25_indexes:
        return []

    bm25 = bm25_indexes[geo]["bm25"]
    docs = bm25_indexes[geo]["docs"]
    query_tokens = tokenize_for_bm25(question)
    scores = bm25.get_scores(query_tokens)

    ranked = sorted(
        [(docs[idx], float(score)) for idx, score in enumerate(scores)],
        key=lambda x: x[1],
        reverse=True,
    )

    filtered = [(doc, score) for doc, score in ranked if score >= settings.bm25_score_threshold]
    return filtered[:k]


def reciprocal_rank_fusion(
    vector_results: List[Tuple[Document, float]],
    bm25_results: List[Tuple[Document, float]],
) -> List[Document]:
    fused_scores = defaultdict(float)
    doc_map: Dict[str, Document] = {}

    for rank, (doc, score) in enumerate(vector_results, start=1):
        key = doc.metadata["chunk_id"]
        fused_scores[key] += 1.0 / (settings.rrf_k + rank)
        doc_map[key] = doc
        doc.metadata["vector_score"] = float(score)

    for rank, (doc, score) in enumerate(bm25_results, start=1):
        key = doc.metadata["chunk_id"]
        fused_scores[key] += 1.0 / (settings.rrf_k + rank)
        doc_map[key] = doc
        doc.metadata["bm25_score"] = float(score)

    ranked_keys = sorted(fused_scores.keys(), key=lambda k: fused_scores[k], reverse=True)

    fused_docs: List[Document] = []
    for key in ranked_keys:
        doc = doc_map[key]
        doc.metadata["rrf_score"] = float(fused_scores[key])
        fused_docs.append(doc)

    return fused_docs


def rerank_documents(question: str, docs: List[Document], reranker_model, top_n: int) -> List[Document]:
    if not docs or reranker_model is None:
        return docs[:top_n]

    pairs = [
        [question, doc.page_content[: settings.reranker_max_chars]]
        for doc in docs
    ]
    scores = reranker_model.predict(pairs)

    ranked = sorted(
        zip(docs, scores),
        key=lambda x: float(x[1]),
        reverse=True,
    )

    reranked_docs: List[Document] = []
    for doc, score in ranked[:top_n]:
        doc.metadata["rerank_score"] = float(score)
        reranked_docs.append(doc)

    return reranked_docs

In [19]:
def retrieve_documents(
    question: str,
    stores: Dict[str, FAISS],
    bm25_indexes: Dict[str, Dict[str, Any]],
    k: int = settings.top_k,
    fetch_k: int = settings.retrieval_fetch_k,
) -> Tuple[List[str], List[Document], Dict[str, Any]]:
    requested_geos = detect_geos_from_query(question)
    target_geos = requested_geos if requested_geos else list(stores.keys())

    diagnostics: Dict[str, Any] = {"geos": target_geos, "by_geo": {}}
    all_candidates: List[Document] = []

    for geo in target_geos:
        if geo not in stores:
            continue

        vector_raw = stores[geo].similarity_search_with_score(question, k=fetch_k)
        vector_filtered = [(doc, score) for doc, score in vector_raw if score <= settings.score_threshold]

        bm25_filtered: List[Tuple[Document, float]] = []
        if settings.use_hybrid_retrieval:
            bm25_filtered = bm25_search(question, geo, bm25_indexes, k=fetch_k)

        if settings.use_hybrid_retrieval:
            fused_docs = reciprocal_rank_fusion(vector_filtered, bm25_filtered)
        else:
            fused_docs = [doc for doc, _ in vector_filtered]

        diagnostics["by_geo"][geo] = {
            "vector_raw_count": len(vector_raw),
            "vector_pass_threshold_count": len(vector_filtered),
            "bm25_count": len(bm25_filtered),
            "fused_count": len(fused_docs),
        }

        all_candidates.extend(fused_docs)

    # de-duplicate retrieval candidates
    deduped_candidates: List[Document] = []
    seen = set()
    for doc in all_candidates:
        key = doc.metadata.get("chunk_id")
        if key not in seen:
            seen.add(key)
            deduped_candidates.append(doc)

    reranked_docs = rerank_documents(question, deduped_candidates, reranker, top_n=k)

    diagnostics["candidate_count_before_rerank"] = len(deduped_candidates)
    diagnostics["final_count"] = len(reranked_docs)

    logger.info(
        "Retrieval | geos=%s question=%s final_count=%s diagnostics=%s",
        target_geos,
        question,
        len(reranked_docs),
        diagnostics,
    )

    return target_geos, reranked_docs, diagnostics

## Prompt templates

In [20]:
STANDARD_SYSTEM_PROMPT = (
    "You are a retrieval-augmented assistant. "
    "Answer only from the supplied context. "
    "Do not use outside knowledge. "
    "If the answer is not present in the context, reply exactly with: "
    "'I do not have the answer in the provided documents.'"
)

COMPARISON_SYSTEM_PROMPT = (
    "You are a retrieval-augmented assistant. "
    "Answer only from the supplied context. "
    "Do not use outside knowledge. "
    "The user is asking a comparison across GEOs. "
    "Structure the answer using these sections when the context supports it: "
    "1) APAC, 2) EMEA, 3) AMER, 4) Similarities, 5) Differences. "
    "Only include sections that are supported by the provided context. "
    "If the answer is not present in the context, reply exactly with: "
    "'I do not have the answer in the provided documents.'"
)


def format_context(docs: List[Document]) -> str:
    blocks = []
    for i, doc in enumerate(docs[: settings.top_k], start=1):
        source = doc.metadata.get("file_name", "unknown")
        page = doc.metadata.get("page", "unknown")
        geo = doc.metadata.get("geo", "unknown")
        text = doc.page_content[: settings.max_chars_per_chunk_in_prompt]
        blocks.append(
            f"[Chunk {i}]\n"
            f"GEO: {geo}\n"
            f"Source: {source}\n"
            f"Page: {page}\n"
            f"Content:\n{text}"
        )
    return "\n\n---\n\n".join(blocks)

## Structured source output

In [21]:
def build_structured_sources(docs: List[Document]) -> Dict[str, List[Dict[str, Any]]]:
    by_geo: Dict[str, List[Dict[str, Any]]] = defaultdict(list)

    for doc in docs:
        by_geo[doc.metadata.get("geo", "UNKNOWN")].append(
            {
                "file_name": doc.metadata.get("file_name"),
                "page": doc.metadata.get("page"),
                "chunk_id": doc.metadata.get("chunk_id"),
                "loader": doc.metadata.get("loader"),
                "vector_score": doc.metadata.get("vector_score"),
                "bm25_score": doc.metadata.get("bm25_score"),
                "rrf_score": doc.metadata.get("rrf_score"),
                "rerank_score": doc.metadata.get("rerank_score"),
            }
        )

    return dict(by_geo)


def extract_assistant_text(generated_output) -> str:
    generated = generated_output[0]["generated_text"]

    if isinstance(generated, list):
        last_item = generated[-1]
        if isinstance(last_item, dict):
            content = last_item.get("content", "")
            if isinstance(content, str):
                return content.strip()

    return str(generated).strip()

## Final answer function

In [22]:
def answer_question(
    question: str,
    stores: Dict[str, FAISS],
    bm25_indexes: Dict[str, Dict[str, Any]],
) -> Dict[str, Any]:
    geos, docs, diagnostics = retrieve_documents(question, stores, bm25_indexes)

    if not docs:
        return {
            "question": question,
            "geo_used": geos,
            "answer": "I do not have the answer in the provided documents.",
            "retrieval_diagnostics": diagnostics,
            "sources_by_geo": {},
            "flat_sources": [],
        }

    system_prompt = (
        COMPARISON_SYSTEM_PROMPT
        if is_comparison_query(question, geos)
        else STANDARD_SYSTEM_PROMPT
    )

    context = format_context(docs)
    geo_label = ", ".join(geos) if geos else "ALL"

    messages = [
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": (
                f"GEO filter: {geo_label}\n\n"
                f"Context:\n{context}\n\n"
                f"Question: {question}"
            ),
        },
    ]

    outputs = generator(
        messages,
        max_new_tokens=settings.max_new_tokens,
        do_sample=settings.do_sample,
        temperature=settings.temperature,
    )

    answer = extract_assistant_text(outputs)

    flat_sources = [
        {
            "geo": doc.metadata.get("geo"),
            "file_name": doc.metadata.get("file_name"),
            "page": doc.metadata.get("page"),
            "chunk_id": doc.metadata.get("chunk_id"),
            "loader": doc.metadata.get("loader"),
            "vector_score": doc.metadata.get("vector_score"),
            "bm25_score": doc.metadata.get("bm25_score"),
            "rrf_score": doc.metadata.get("rrf_score"),
            "rerank_score": doc.metadata.get("rerank_score"),
        }
        for doc in docs
    ]

    result = {
        "question": question,
        "geo_used": geos,
        "answer": answer,
        "retrieval_diagnostics": diagnostics,
        "sources_by_geo": build_structured_sources(docs),
        "flat_sources": flat_sources,
    }

    logger.info(
        "Answer complete | geos=%s sources=%s question=%s",
        geos,
        len(flat_sources),
        question,
    )
    return result

## Example usage

In [25]:
# Optional: inspect retrieval scores to tune thresholds
inspect_retrieval_scores("What is the transformer policy for APAC?", geo_vectorstores, k=5)

Query GEOs: ['APAC']

--- APAC ---
1. score=0.9904 | file=2.pdf | page=88 | chunk_id=ac09817f910c93efc010b1b5b5754333
2. score=0.9979 | file=2.pdf | page=65 | chunk_id=8efa38c0526609ed970667c352e083e1
3. score=1.0099 | file=2.pdf | page=92 | chunk_id=2180e7bc858bd7f683fc5193fdf892d4
4. score=1.0308 | file=2.pdf | page=91 | chunk_id=125c5a5fcc4c74ece5805ca1c380d419
5. score=1.0331 | file=2.pdf | page=90 | chunk_id=5e62820ed0526f88df3157e321a609dd


In [26]:
result = answer_question("What is the transformer for APAC?", geo_vectorstores, bm25_indexes)
print(result["answer"])
print(json.dumps(result["sources_by_geo"], indent=2))

2026-03-31 00:11:43 | INFO | Retrieval | geos=['APAC'] question=What is the transformer for APAC? final_count=4 diagnostics={'geos': ['APAC'], 'by_geo': {'APAC': {'vector_raw_count': 10, 'vector_pass_threshold_count': 10, 'bm25_count': 10, 'fused_count': 20}}, 'candidate_count_before_rerank': 20, 'final_count': 4}
2026-03-31 00:13:55 | INFO | Answer complete | geos=['APAC'] sources=4 question=What is the transformer for APAC?


I do not have the answer in the provided documents.
{
  "APAC": [
    {
      "file_name": "5deb6b48-5d7f-4a37-a361-d01317ac4bb7.pdf",
      "page": 2,
      "chunk_id": "5aae2212b2396dd5c2a895b283a807d4",
      "loader": "pypdf",
      "vector_score": null,
      "bm25_score": 4.086658973741207,
      "rrf_score": 0.014492753623188406,
      "rerank_score": -3.9195289611816406
    },
    {
      "file_name": "2.pdf",
      "page": 57,
      "chunk_id": "569f1951685aed03acbc6b43ba0edff5",
      "loader": "pypdf",
      "vector_score": null,
      "bm25_score": 5.833302119071228,
      "rrf_score": 0.01639344262295082,
      "rerank_score": -3.931180000305176
    },
    {
      "file_name": "5deb6b48-5d7f-4a37-a361-d01317ac4bb7.pdf",
      "page": 7,
      "chunk_id": "aa443d3143af7f55a36b8ebf1a980a8d",
      "loader": "pypdf",
      "vector_score": null,
      "bm25_score": 4.069367631106865,
      "rrf_score": 0.014285714285714285,
      "rerank_score": -4.86899995803833
    },
    {


In [27]:
result = answer_question("Compare the transformer for APAC and AMER", geo_vectorstores, bm25_indexes)
print(result["answer"])
print(json.dumps(result["sources_by_geo"], indent=2))
print(json.dumps(result["retrieval_diagnostics"], indent=2))

2026-03-31 00:13:56 | INFO | Retrieval | geos=['APAC', 'AMER'] question=Compare the transformer for APAC and AMER final_count=4 diagnostics={'geos': ['APAC', 'AMER'], 'by_geo': {'APAC': {'vector_raw_count': 10, 'vector_pass_threshold_count': 10, 'bm25_count': 10, 'fused_count': 20}, 'AMER': {'vector_raw_count': 10, 'vector_pass_threshold_count': 5, 'bm25_count': 10, 'fused_count': 14}}, 'candidate_count_before_rerank': 34, 'final_count': 4}
2026-03-31 00:17:22 | INFO | Answer complete | geos=['APAC', 'AMER'] sources=4 question=Compare the transformer for APAC and AMER


1) APAC, AMER
2) The context provided does not contain specific information about the differences or similarities in transformer measurements or tests conducted in APAC versus AMER. The content available focuses more on the theoretical aspects and testing procedures for transformers rather than regional differences.
3) Similarities
   - Both regions discuss the short circuit test on transformers, where the ammeter reading gives the primary equivalent of full load current (Isc).
   - Both regions mention that the voltage applied for full load current is very small compared to the rated voltage, allowing core loss to be neglected.
   - Both regions provide equations related to calculating copper losses and equivalent reactance of the transformer.
4) Differences
   - No specific differences are mentioned in the provided context.
5) Conclusion
   I do not have the answer in the provided documents.
{
  "AMER": [
    {
      "file_name": "11.pdf",
      "page": 3,
      "chunk_id": "9eb845ff

## Notes for reruns

When you rerun the ingestion cells:

- new PDFs are added
- updated PDFs are reprocessed
- deleted PDFs are removed
- only the touched GEO indexes are rebuilt

That avoids rebuilding the entire corpus every time.

## Tuning suggestions

If the model is still heavy for your machine:

```python
settings.llm_model = "Qwen/Qwen2.5-1.5B-Instruct"
settings.top_k = 3
settings.max_new_tokens = 96
settings.max_chars_per_chunk_in_prompt = 900
```

If retrieval is too permissive:

```python
settings.score_threshold = 1.0
settings.bm25_score_threshold = 0.10
```

If retrieval is too strict:

```python
settings.score_threshold = 1.3
settings.bm25_score_threshold = 0.01
```

## Final design decisions locked in now

These are the choices worth keeping **as-is** to avoid rework later:

1. **One FAISS index per GEO**
   - simpler GEO filtering
   - simpler incremental rebuilds
   - easier operational debugging

2. **Persistent JSONL chunk stores per GEO**
   - required for sane delete/update handling
   - lets you rebuild only affected GEO indexes
   - avoids depending on FAISS internals for document removal

3. **Registry-driven incremental ingestion**
   - tracks path, GEO, mtime, size, sha256, doc_id
   - distinguishes new / updated / deleted PDFs
   - limits rebuild work to touched GEOs only

4. **Hybrid retrieval before generation**
   - vector retrieval for semantic matches
   - BM25 for exact phrases / IDs / policy names
   - reranker on top for final relevance

5. **Threshold before LLM**
   - weak retrieval should stop before generation
   - prevents forced answers from poor context

6. **Structured source output**
   - keep flat sources and grouped-by-GEO sources
   - makes debugging and downstream UI integration easier later

7. **OCR as fallback, not default**
   - use `PyPDFLoader` first
   - OCR only when extraction quality is poor
   - avoids unnecessary OCR cost on normal text PDFs

These are the right defaults for the current design, so I would keep them fixed unless your data shape changes materially.

## Validation helpers for incremental ingestion

These cells let you verify the “no full rebuild” behavior after:
- adding a PDF
- updating a PDF
- deleting a PDF

The goal is to prove that only the touched GEO indexes are rebuilt.

In [ ]:
def get_geo_artifact_snapshot() -> Dict[str, Dict[str, Any]]:
    snapshot: Dict[str, Dict[str, Any]] = {}

    for geo in settings.geos:
        index_dir = geo_index_path(geo)
        chunk_path = chunk_store_path(geo)

        index_mtime = None
        if index_dir.exists():
            mtimes = [p.stat().st_mtime for p in index_dir.rglob("*") if p.is_file()]
            index_mtime = max(mtimes) if mtimes else index_dir.stat().st_mtime

        chunk_count = 0
        if chunk_path.exists():
            with chunk_path.open("r", encoding="utf-8") as f:
                chunk_count = sum(1 for line in f if line.strip())

        snapshot[geo] = {
            "index_exists": index_dir.exists(),
            "index_mtime": index_mtime,
            "chunk_store_exists": chunk_path.exists(),
            "chunk_count": chunk_count,
        }

    return snapshot


def print_geo_snapshot(snapshot: Dict[str, Dict[str, Any]], title: str = "Snapshot") -> None:
    print(f"\n{title}")
    print("-" * len(title))
    for geo, info in snapshot.items():
        print(
            f"{geo}: index_exists={info['index_exists']}, "
            f"index_mtime={info['index_mtime']}, "
            f"chunk_store_exists={info['chunk_store_exists']}, "
            f"chunk_count={info['chunk_count']}"
        )


def compare_snapshots(before: Dict[str, Dict[str, Any]], after: Dict[str, Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    comparison: Dict[str, Dict[str, Any]] = {}

    for geo in settings.geos:
        b = before.get(geo, {})
        a = after.get(geo, {})
        comparison[geo] = {
            "index_changed": b.get("index_mtime") != a.get("index_mtime"),
            "chunk_count_before": b.get("chunk_count"),
            "chunk_count_after": a.get("chunk_count"),
            "chunk_count_delta": (a.get("chunk_count") or 0) - (b.get("chunk_count") or 0),
        }

    return comparison


def print_snapshot_comparison(comparison: Dict[str, Dict[str, Any]]) -> None:
    print("\nSnapshot comparison")
    print("-------------------")
    for geo, info in comparison.items():
        print(
            f"{geo}: index_changed={info['index_changed']}, "
            f"chunk_count_before={info['chunk_count_before']}, "
            f"chunk_count_after={info['chunk_count_after']}, "
            f"chunk_count_delta={info['chunk_count_delta']}"
        )

## Step A — capture a baseline

Run this **before** you add, update, or delete a PDF.

In [ ]:
baseline_registry = load_registry()
baseline_snapshot = get_geo_artifact_snapshot()

print(f"Registry entries: {len(baseline_registry)}")
print_geo_snapshot(baseline_snapshot, title="Baseline snapshot")

## Step B — make one controlled change outside the notebook

Do **one** of these in your `data/` folders:

1. Add a new PDF to one GEO folder  
2. Replace or edit an existing PDF in one GEO folder  
3. Delete one PDF from one GEO folder

Keep it to a single GEO for the first validation run.

## Step C — rerun incremental ingestion and verify only one GEO changed

In [ ]:
registry = load_registry()
inventory = scan_pdf_inventory(settings.data_dir)
change_plan = plan_ingestion_changes(inventory, registry)
log_change_plan(change_plan)

geo_vectorstores = rebuild_touched_geos_only(change_plan, registry, embeddings)

if not change_plan["touched_geos"]:
    geo_vectorstores = load_geo_vectorstores(embeddings)

after_snapshot = get_geo_artifact_snapshot()

print("Touched GEOs:", change_plan["touched_geos"])
print_geo_snapshot(after_snapshot, title="After-change snapshot")

comparison = compare_snapshots(baseline_snapshot, after_snapshot)
print_snapshot_comparison(comparison)

## Step D — expected result

For a single-GEO change, you should see:

- `change_plan["touched_geos"]` contains only that GEO
- only that GEO shows `index_changed=True`
- only that GEO usually shows a non-zero `chunk_count_delta`
- untouched GEOs should remain unchanged

That confirms incremental ingestion is working as intended.

## Operational guardrails to keep fixed

To avoid rework, keep these rules fixed now:

- keep **one GEO per folder** at ingestion time
- keep **registry + chunk stores + FAISS** together under `storage/`
- do **not** try to remove or update documents directly inside FAISS without rebuilding the touched GEO index from the chunk store
- keep **OCR fallback optional** and dependency-based
- keep **reranker toggleable** with `settings.use_reranker`
- keep **hybrid retrieval toggleable** with `settings.use_hybrid_retrieval`

Those choices give you the cleanest path for maintenance and debugging.